In [3]:
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

RAW_PATH = "../data/raw/loans.csv"
CLEAN_PATH = "../data/processed/loans_clean.csv"

# Left side: a standard name used in this project. Right side: YOUR column name.
COLUMNS = {
    "income": "revenue",
    "dti": "dti_n",
    "loan_amount": "loan_amnt",
    "credit_score": "fico_n",
    "employment": "emp_length",
    "purpose": "purpose",
    "home_ownership": "home_ownership_n",
    "state": "addr_state",
    "issue_date": "issue_d",
    "outcome": "Default",
}

# How to turn the outcome column into default (1) or not default (0).
# Your 'Default' column is already numeric (1 = default, 0 = paid).
DEFAULT_VALUES = [1]
NOT_DEFAULT_VALUES = [0]

CREDIT_SCORE_RANGE = (300, 850)     # valid range of a FICO score

In [4]:
raw = pd.read_csv(RAW_PATH, low_memory=False)

# Keep only the ten columns we need and give them standard names
df = raw[list(COLUMNS.values())].rename(columns={v: k for k, v in COLUMNS.items()})
df = df.rename(columns={
    "income": "Income", "dti": "DTI", "loan_amount": "LoanAmount", "credit_score": "CreditScore",
    "employment": "EmploymentLength", "purpose": "Purpose", "home_ownership": "HomeOwnership",
    "state": "State", "issue_date": "IssueDate", "outcome": "Outcome",
}).copy()

# A log so we can report exactly how many rows each step removed
cleaning_log = []


def record(step):
    cleaning_log.append({"step": step, "rows_left": len(df)})


record("Start (selected columns only)")
df.head()

,Income,DTI,LoanAmount,CreditScore,EmploymentLength,Purpose,HomeOwnership,State,IssueDate,Outcome
0,55000.0,5.91,3600,677.0,10+ years,debt_consolidation,MORTGAGE,PA,Dec-2015,0
1,65000.0,16.06,24700,717.0,10+ years,small_business,MORTGAGE,SD,Dec-2015,0
2,71000.0,13.85,20000,697.0,10+ years,home_improvement,MORTGAGE,IL,Dec-2015,0
3,104433.0,25.37,10400,697.0,3 years,major_purchase,MORTGAGE,PA,Dec-2015,0
4,34000.0,10.20,11950,692.0,4 years,debt_consolidation,RENT,GA,Dec-2015,0


In [5]:
is_default = df["Outcome"].isin(DEFAULT_VALUES)
is_paid = df["Outcome"].isin(NOT_DEFAULT_VALUES)

print("Default:", is_default.sum(), "| Not default:", is_paid.sum(),
      "| Unresolved or other:", (~(is_default | is_paid)).sum())
print(df.loc[~(is_default | is_paid), "Outcome"].value_counts(dropna=False))

df = df[is_default | is_paid].copy()
df["Default"] = df["Outcome"].isin(DEFAULT_VALUES).astype(int)
df = df.drop(columns="Outcome")
record("Kept only resolved loans (default or fully paid)")

print("Default rate:", round(df["Default"].mean(), 3))

Default: 269249 | Not default: 1078432 | Unresolved or other: 0
Series([], Name: count, dtype: int64)
Default rate: 0.2


In [6]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
record("Removed duplicate rows")

Duplicates: 0


In [7]:
for column in ["Income", "DTI", "LoanAmount", "CreditScore"]:
    before = df[column].isna().sum()
    df[column] = pd.to_numeric(df[column], errors="coerce")
    print(f"{column:12s} became missing while converting: {df[column].isna().sum() - before}")

Income       became missing while converting: 0
DTI          became missing while converting: 0
LoanAmount   became missing while converting: 0
CreditScore  became missing while converting: 0


In [8]:
def parse_employment_length(value):
    """'10+ years' -> 10, '< 1 year' -> 0, '3 years' -> 3, missing stays missing."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).lower()
    if "<" in text:
        return 0.0
    digits = "".join(character for character in text if character.isdigit())
    return float(digits) if digits else np.nan


df["EmploymentLength"] = df["EmploymentLength"].apply(parse_employment_length)
print(df["EmploymentLength"].value_counts(dropna=False).sort_index())

EmploymentLength
0.0     108517
1.0      88841
2.0     122091
3.0     107862
4.0      80761
5.0      84326
6.0      62877
7.0      59724
8.0      60808
9.0      51019
10.0    442667
NaN      78188
Name: count, dtype: int64


In [9]:
def parse_employment_length(value):
    """'10+ years' -> 10, '< 1 year' -> 0, '3 years' -> 3, missing stays missing."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).lower()
    if "<" in text:
        return 0.0
    digits = "".join(character for character in text if character.isdigit())
    return float(digits) if digits else np.nan


df["EmploymentLength"] = df["EmploymentLength"].apply(parse_employment_length)
print(df["EmploymentLength"].value_counts(dropna=False).sort_index())

EmploymentLength
0.0     108517
1.0      88841
2.0     122091
3.0     107862
4.0      80761
5.0      84326
6.0      62877
7.0      59724
8.0      60808
9.0      51019
10.0    442667
NaN      78188
Name: count, dtype: int64


In [11]:
df["IssueDate"] = pd.to_datetime(df["IssueDate"], errors="coerce")
print("Unreadable dates:", df["IssueDate"].isna().sum())

df = df.dropna(subset=["IssueDate"]).reset_index(drop=True)
df["IssueYear"] = df["IssueDate"].dt.year
record("Removed rows with unreadable issue dates")

print(df["IssueYear"].value_counts().sort_index())

Unreadable dates: 0
IssueYear
2007       599
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134804
2014    223102
2015    375543
2016    293057
2017    169117
2018     56160
Name: count, dtype: int64


C:\Users\yoges\AppData\Local\Temp\ipykernel_22800\1204362003.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["IssueDate"] = pd.to_datetime(df["IssueDate"], errors="coerce")


In [10]:
low, high = CREDIT_SCORE_RANGE

impossible = {
    "Income": df["Income"] <= 0,
    "LoanAmount": df["LoanAmount"] <= 0,
    "DTI": df["DTI"] < 0,
    "CreditScore": (df["CreditScore"] < low) | (df["CreditScore"] > high),
}

for column, mask in impossible.items():
    print(f"{column:12s} impossible values set to missing: {mask.sum()}")
    df.loc[mask, column] = np.nan

Income       impossible values set to missing: 0
LoanAmount   impossible values set to missing: 0
DTI          impossible values set to missing: 0
CreditScore  impossible values set to missing: 0


In [12]:
for column in ["Purpose", "HomeOwnership"]:
    df[column] = df[column].astype("string").str.strip().str.lower().fillna("unknown")

df["State"] = df["State"].astype("string").str.strip().str.upper().fillna("UNKNOWN")

print(df["Purpose"].value_counts().head(10))

Purpose
debt_consolidation    781206
credit_card           295551
home_improvement       87684
other                  78263
major_purchase         29543
medical                15606
small_business         15575
car                    14647
moving                  9520
vacation                9083
Name: count, dtype: int64[pyarrow]


In [13]:
def iqr_outliers(series):
    """Count values far outside the typical range (the IQR rule)."""
    q1, q3 = series.quantile([0.25, 0.75])
    spread = q3 - q1
    return int(((series < q1 - 1.5 * spread) | (series > q3 + 1.5 * spread)).sum())


for column in ["Income", "DTI", "LoanAmount"]:
    print(f"{column:12s} IQR outliers (kept): {iqr_outliers(df[column].dropna())}")

print("\nMissing values (%):")
print((df.isna().mean() * 100).round(2).sort_values(ascending=False))

Income       IQR outliers (kept): 62998
DTI          IQR outliers (kept): 5506
LoanAmount   IQR outliers (kept): 7131

Missing values (%):
EmploymentLength    5.8
DTI                 0.0
Income              0.0
LoanAmount          0.0
CreditScore         0.0
Purpose             0.0
HomeOwnership       0.0
State               0.0
IssueDate           0.0
Default             0.0
IssueYear           0.0
dtype: float64


In [14]:
df = df.sort_values("IssueDate").reset_index(drop=True)
record("Final clean data")

assert df.duplicated().sum() == 0, "duplicates remain"
assert df["Default"].isin([0, 1]).all(), "Default must be 0 or 1"
assert df["IssueDate"].notna().all(), "dates are missing"
assert (df["Income"].dropna() > 0).all(), "non-positive income remains"
assert df["CreditScore"].dropna().between(low, high).all(), "credit score out of range"
print("All checks passed")

os.makedirs("../data/processed", exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("Saved:", CLEAN_PATH, df.shape)

pd.DataFrame(cleaning_log)

All checks passed
Saved: ../data/processed/loans_clean.csv (1347681, 11)


,step,rows_left
0,Start (selected columns only),1347681
1,Kept only resolved loans (default or fully paid),1347681
2,Removed duplicate rows,1347681
3,Removed rows with unreadable issue dates,1347681
4,Final clean data,1347681
